# 01 — EDA: GA4 Ecommerce Verhaltensdaten

**Person:** A | **Hypothese:** Datengrundlage für H2

**Ziel:**
- GA4-Daten laden und strukturell verstehen
- Session-Funnel analysieren
- User-Verhaltensmuster identifizieren
- RFM-Grundlage vorbereiten

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data.loader import load_ga4_events
from src.utils.helpers import save_figure

print('Setup OK')

Setup OK


## 1. Daten laden

In [18]:
# GA4 Events laden
pd.set_option("display.max_columns", 100)

path = r"C:\Users\shiva\OneDrive\Desktop\weiter bildung\projekt\ki-marketing-personalisierung-main\data\raw\ga4_ecommerce\GA4_Ecommer.csv"  # Pfad anpassen
df = pd.read_csv(path)

df.shape, df.dtypes.head()

df = load_ga4_events('GA4_Ecommer.csv')  # Pfad anpassen
df.head(5)

,event_date,event_timestamp,event_name,user_pseudo_id,platform,device_category,country
0,20210131,1.612070e+15,page_view,1026454.427,WEB,mobile,United States
1,20210131,1.612070e+15,scroll,1026454.427,WEB,mobile,United States
2,20210131,1.612070e+15,page_view,1026454.427,WEB,mobile,United States
3,20210131,1.612070e+15,user_engagement,1026454.427,WEB,mobile,United States
4,20210131,1.612070e+15,session_start,1026454.427,WEB,mobile,United States


## 2. Datenstruktur & Qualität

In [19]:
df.info()
df.describe()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26489 entries, 0 to 26488
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   event_date       26489 non-null  int64  
 1   event_timestamp  26489 non-null  float64
 2   event_name       26489 non-null  object 
 3   user_pseudo_id   26489 non-null  float64
 4   platform         26489 non-null  object 
 5   device_category  26489 non-null  object 
 6   country          26489 non-null  object 
dtypes: float64(2), int64(1), object(4)
memory usage: 1.4+ MB


event_date         0
event_timestamp    0
event_name         0
user_pseudo_id     0
platform           0
device_category    0
country            0
dtype: int64

## 3. Event-Verteilung

In [20]:
event_counts = df['event_name'].value_counts()
print(event_counts.head(20))

event_name
page_view              9498
user_engagement        5005
scroll                 2870
session_start          2760
first_visit            2127
view_item              1829
view_promotion         1190
add_to_cart             295
select_item             237
begin_checkout          234
view_search_results     198
add_shipping_info       100
select_promotion         71
add_payment_info         53
purchase                 19
click                     3
Name: count, dtype: int64


## 4. Conversion Funnel

In [21]:
# Funnel: page_view → view_item → add_to_cart → begin_checkout → purchase
funnel_events = ['page_view', 'view_item', 'add_to_cart', 'begin_checkout', 'purchase']
funnel = df[df['event_name'].isin(funnel_events)]['event_name'].value_counts()

## 5. User-Aktivität (RFM-Vorbereitung)

In [22]:
# Purchase-Events filtern
purchases = df[df['event_name'] == 'purchase'].copy()
purchases.head()

,event_date,event_timestamp,event_name,user_pseudo_id,platform,device_category,country
23538,20210131,1.612080e+15,purchase,1617434.154,WEB,mobile,Singapore
23540,20210131,1.612081e+15,purchase,1617434.154,WEB,mobile,Singapore
23580,20210131,1.612057e+15,purchase,2465265.724,WEB,mobile,United States
23601,20210131,1.612097e+15,purchase,3680421.421,WEB,desktop,India
23615,20210131,1.612089e+15,purchase,4696219.940,WEB,mobile,United States


## 6. Key Findings (Basierend auf den echten GA4-Daten)

Finding 1:  
Der größte Drop‑off im Funnel erfolgt zwischen view_item (1829 Events) und add_to_cart (295 Events).
Das bedeutet, dass nur etwa 16 % der Nutzer, die ein Produkt ansehen, es tatsächlich in den Warenkorb legen.
→ Hohes Interesse, aber geringe Kaufintention.

Finding 2:  
Die Conversion von add_to_cart (295) zu begin_checkout (234) ist relativ hoch (≈ 79 %).
Nutzer, die ein Produkt in den Warenkorb legen, zeigen also eine starke Kaufbereitschaft.

Finding 3:  
Die finale Conversion von begin_checkout (234) zu purchase (19) ist sehr niedrig (≈ 8 %).
Dies deutet auf Probleme im Checkout‑Prozess hin (z. B. fehlende Zahlungsmethoden, Versandkosten, Usability‑Probleme, Ladezeiten).

Finding 4:  
Der Anteil der purchase-Events (19 von 26.489 Gesamt‑Events) ist extrem gering.
Dies weist auf eine insgesamt niedrige E‑Commerce‑Performance hin und zeigt, dass der Funnel stark optimierungsbedürftig ist.

Finding 5:  
Die meisten Events stammen von mobile‑Geräten.
Das bedeutet, dass Mobile‑UX‑Optimierung besonders wichtig ist, da die Mehrheit der Nutzer mobil interagiert.